# Placement V4 cumulative 75-game development fit

이 Notebook은 temporal holdout에서 채택한 V4 representation을 바꾸지 않고 cumulative accepted cohort로 development model을 batch fit하는 실행 가능한 checkpoint다. 결과는 held-out 성능 증거, final trainer 또는 production artifact가 아니다.


## 1. 실험 질문과 경계

**왜 실행하는가:** 이미 검증된 V4 정책을 75 accepted game 전체에 적용해 scaler와 coefficient를 결정적으로 만들 수 있는지 확인한다.

**입력:** immutable cumulative dataset의 train/validation JSONL을 합친 accepted rows.

**계산:** V4 physical-class pair, raw-candidate StandardScaler, symmetric sample과 decision-balanced weight로 LogisticRegression 하나를 fit한다.

**읽는 법:** Smoke는 같은 경로를 소규모 game subset으로 검증한다. Full만 75게임 checkpoint다.

**보류 조건:** checksum, schema, V4, weight, convergence 또는 source immutability가 깨지면 checkpoint를 만들지 않는다. LOGO, V0~V3 비교, ranking metric과 production integration은 범위 밖이다.


In [ ]:
from __future__ import annotations

import hashlib
import io
import json
import os
import platform
import secrets
import warnings
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import scipy
import sklearn
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

EXPERIMENT_ID = "placement-v4-cumulative-75g-development-fit"
REPRESENTATION = "V4"
EXPECTED_ACCEPTED_GAME_COUNT = 75
EXPECTED_DECISION_COUNT = 3_480
EXPECTED_CANDIDATE_COUNT = 117_805
MODEL_CONFIG = {"solver": "lbfgs", "fitIntercept": False, "classWeight": None, "tolerance": 1e-12, "maxIterations": 10_000, "l2": 0.01}


## 2. 환경, 경로와 checksum

**왜 실행하는가:** 학습 전에 정확한 immutable source를 고정한다.

**입력:** project-relative dataset 네 파일과 선택적인 report 환경변수.

**계산:** SHA-256과 환경 버전을 기록한다. Dataset은 export하거나 복사하지 않는다.

**읽는 법:** expected와 actual checksum이 완전히 같아야 한다. Report에는 사용자 절대 경로를 넣지 않는다.

**보류 조건:** 파일이 없거나 checksum이 다르면 JSONL을 읽기 전에 중단한다.


In [ ]:
def find_repository_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "package.json").is_file() and (candidate / "python_tests").is_dir():
            return candidate
    raise AssertionError("REPOSITORY_ROOT_NOT_FOUND")


def resolve_config_path(value: str | Path, root: Path) -> Path:
    path = Path(value)
    return path if path.is_absolute() else root / path


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


ROOT = find_repository_root(Path.cwd())
DATASET_RELATIVE = Path("data/datasets/human-behavior-v2-temporal-cumulative-75g-20260718")
DATASET = resolve_config_path(os.environ.get("PLACEMENT_V4_DEVELOPMENT_DATASET", str(DATASET_RELATIVE)), ROOT)
REPORT_PATH = resolve_config_path(os.environ.get("PLACEMENT_V4_DEVELOPMENT_REPORT", "/private/tmp/placement-v4-cumulative-75g-development-fit-smoke.json"), ROOT)
DATASET_FILES = ["manifest.json", "train.jsonl", "validation.jsonl", "rejected.jsonl"]
EXPECTED_SHA256 = {
    "manifest.json": "dbf1dd5f30177abc3b44d7bdcc7e21aaf2f8a3d0015a96ba6bdfd9506c71487a",
    "train.jsonl": "0a71b121514016721d50bda5e01b298002fe8820e0a110f6b62945b4f1550120",
    "validation.jsonl": "65511767072d46484a5e4e15821bde2ee91957d02b1e16af35ec5e9da2c5274d",
    "rejected.jsonl": "84c93b3b1d8e4d82eb4dcf094bab28c35d8834dedf9566bfbe72b008b47b1085",
}
for name in DATASET_FILES:
    assert (DATASET / name).is_file(), f"DATASET_FILE_REQUIRED:{name}"
source_checksums_before = {name: sha256_file(DATASET / name) for name in DATASET_FILES}
assert source_checksums_before == EXPECTED_SHA256, "CUMULATIVE_DATASET_CHECKSUM_MISMATCH"
ENVIRONMENT = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__, "scikitLearn": sklearn.__version__, "platform": platform.system(), "architecture": platform.machine()}
display(pd.DataFrame([{"dataset": DATASET_RELATIVE.as_posix(), **source_checksums_before}]))


## 3. JSONL과 accepted cohort 계약

**왜 실행하는가:** exporter split은 provenance일 뿐 development fit의 holdout이 아니므로 accepted train/validation rows를 합친다.

**입력:** manifest, train, validation, rejected.

**계산:** UTF-8/JSON, schema v2, Human source, boolean label, manifest row count와 decision당 정확히 하나의 selected candidate를 검증한다.

**읽는 법:** manifest lifecycle game count 76은 입력 lifecycle 수이고 accepted training game count 75와 구분한다.

**보류 조건:** actual accepted count가 75/3,480/117,805와 다르거나 non-Human row가 있으면 중단한다.


In [ ]:
def read_json_object(path: Path) -> dict[str, Any]:
    value = json.loads(path.read_bytes().decode("utf-8", errors="strict"))
    assert isinstance(value, dict), f"JSON_OBJECT_REQUIRED:{path.name}"
    return value


def read_jsonl_frame(path: Path) -> pd.DataFrame:
    text = path.read_bytes().decode("utf-8", errors="strict")
    lines = text.splitlines()
    assert lines and all(line.strip() for line in lines), f"JSONL_LINE_REQUIRED:{path.name}"
    for line_number, line in enumerate(lines, 1):
        assert isinstance(json.loads(line), dict), f"JSON_OBJECT_LINE_REQUIRED:{path.name}:{line_number}"
    frame = pd.read_json(io.StringIO(text), lines=True, dtype=False)
    assert len(frame) == len(lines), f"ROW_COUNT_MISMATCH:{path.name}"
    return frame


MANIFEST = read_json_object(DATASET / "manifest.json")
train_rows = read_jsonl_frame(DATASET / "train.jsonl").assign(originalSplit="train")
validation_rows = read_jsonl_frame(DATASET / "validation.jsonl").assign(originalSplit="validation")
rejected_rows = read_jsonl_frame(DATASET / "rejected.jsonl")
assert len(train_rows) == MANIFEST["trainSampleCount"]
assert len(validation_rows) == MANIFEST["validationSampleCount"]
assert len(rejected_rows) == MANIFEST["rejectedPlacementLogCount"]
all_candidates = pd.concat([train_rows, validation_rows], ignore_index=True, copy=True)
assert len(all_candidates) == MANIFEST["sampleCountBySource"]["human"] == EXPECTED_CANDIDATE_COUNT
REQUIRED_COLUMNS = {"datasetSchemaVersion", "gameId", "playerId", "pieceIndex", "source", "candidatePlacement", "candidateResult", "candidateMetricsAfter", "metricDelta", "selectedBySource", "metadata"}
assert not REQUIRED_COLUMNS.difference(all_candidates.columns), "REQUIRED_DATASET_COLUMN_MISSING"
assert (all_candidates["datasetSchemaVersion"] == 2).all()
assert (all_candidates["source"] == "human").all(), "NON_HUMAN_SOURCE"
assert all_candidates["selectedBySource"].map(lambda value: isinstance(value, (bool, np.bool_))).all(), "BOOLEAN_SELECTED_REQUIRED"
assert MANIFEST["sourceFilter"] == "human" and MANIFEST["selectionPolicy"] == "all-completed"
assert MANIFEST["jsonParseFailureCount"] == 0


## 4. Smoke/full training cohort

**왜 실행하는가:** full과 smoke가 같은 ordering과 학습 함수를 사용하되 자동 작업에서는 작은 경로만 확인한다.

**입력:** accepted gameId와 `PLACEMENT_V4_MAX_TRAINING_GAMES`.

**계산:** Unicode code-point 순서의 처음 N game을 고른다. N=75일 때만 full cohort다.

**읽는 법:** Smoke coefficient는 버리는 진단이며 full coefficient로 해석하지 않는다.

**보류 조건:** N이 1~75 범위를 벗어나거나 full에서 count가 고정 계약과 다르면 중단한다.


In [ ]:
def codepoint_key(value: Any) -> tuple[int, ...]:
    return tuple(ord(character) for character in str(value))


def create_decision_key(row: pd.Series) -> str:
    return f"{row['gameId']}:{row['playerId']}:{int(row['pieceIndex'])}"


def create_placement_key(row: pd.Series) -> str:
    placement = row["candidatePlacement"]
    return f"{placement['pieceType']}:{int(placement['rotation'])}:{int(placement['row'])}:{int(placement['col'])}"


all_candidates["decisionKey"] = [create_decision_key(row) for _, row in all_candidates.iterrows()]
all_candidates["placementKey"] = [create_placement_key(row) for _, row in all_candidates.iterrows()]
decision_contract = all_candidates.groupby("decisionKey", sort=False).agg(selectedCount=("selectedBySource", "sum"), gameCount=("gameId", "nunique"), candidateCount=("placementKey", "size"), placementCount=("placementKey", "nunique"))
assert (decision_contract["selectedCount"] == 1).all(), "ONE_SELECTED_CANDIDATE_REQUIRED"
assert (decision_contract["gameCount"] == 1).all() and (decision_contract["candidateCount"] >= 2).all()
assert (decision_contract["candidateCount"] == decision_contract["placementCount"]).all(), "PLACEMENT_KEY_UNIQUE_REQUIRED"
ALL_GAME_IDS = sorted(all_candidates["gameId"].unique(), key=codepoint_key)
assert len(ALL_GAME_IDS) == EXPECTED_ACCEPTED_GAME_COUNT
assert len(decision_contract) == EXPECTED_DECISION_COUNT
MAX_TRAINING_GAMES = int(os.environ.get("PLACEMENT_V4_MAX_TRAINING_GAMES", "2"))
assert 1 <= MAX_TRAINING_GAMES <= EXPECTED_ACCEPTED_GAME_COUNT, "TRAINING_GAME_LIMIT_OUT_OF_RANGE"
TRAINING_GAME_IDS = ALL_GAME_IDS[:MAX_TRAINING_GAMES]
FULL_COHORT_COVERED = len(TRAINING_GAME_IDS) == EXPECTED_ACCEPTED_GAME_COUNT
RUN_MODE = "full" if FULL_COHORT_COVERED else "smoke"
training_candidates = all_candidates.loc[all_candidates["gameId"].isin(TRAINING_GAME_IDS)].copy(deep=True).reset_index(drop=True)
training_decision_count = int(training_candidates["decisionKey"].nunique())
if FULL_COHORT_COVERED:
    assert training_decision_count == EXPECTED_DECISION_COUNT and len(training_candidates) == EXPECTED_CANDIDATE_COUNT
display(pd.DataFrame([{"runMode": RUN_MODE, "availableAcceptedGames": len(ALL_GAME_IDS), "fittedGames": len(TRAINING_GAME_IDS), "fittedDecisions": training_decision_count, "fittedCandidates": len(training_candidates), "manifestLifecycleGames": MANIFEST["totalGameCount"]}]))


## 5. 고정 16-feature와 occupied-cell V4

**왜 실행하는가:** 기존 representation 계약을 재사용하며 새 feature가 섞이지 않았음을 보인다.

**입력:** candidateResult, metricDelta와 candidatePlacement absolute cells.

**계산:** 16-feature vector와 label-free `pieceType + sorted occupied cells` class key를 만든다.

**읽는 법:** selectedBySource는 pair label 식별에만 쓰고 winner, heuristic score와 input sequence는 feature가 아니다.

**보류 조건:** 같은 occupied-cell class의 feature 또는 즉시 결과가 다르면 fit 전에 중단한다.


In [ ]:
FEATURE_NAMES = [
    "candidateResult.linesCleared", "candidateResult.gameOver",
    "metricDelta.aggregateHeight", "metricDelta.maxHeight",
    "metricDelta.holes", "metricDelta.bumpiness",
    *[f"metricDelta.columnHeights[{index}]" for index in range(10)],
]
FORBIDDEN_FEATURES = {"selectedBySource", "metadata.isWinner", "winnerPlayerId", "heuristicBaseline.score", "metadata.inputSequence"}


def extract_feature_vector(row: pd.Series) -> np.ndarray:
    result = row["candidateResult"]
    delta = row["metricDelta"]
    vector = np.asarray([int(result["linesCleared"]), int(bool(result["gameOver"])), float(delta["aggregateHeight"]), float(delta["maxHeight"]), float(delta["holes"]), float(delta["bumpiness"]), *[float(value) for value in delta["columnHeights"]]], dtype=np.float64)
    assert vector.shape == (16,) and np.isfinite(vector).all()
    return vector


def create_equivalence_key(row: pd.Series) -> str:
    placement = row["candidatePlacement"]
    cells = sorted((int(cell["row"]), int(cell["col"])) for cell in placement["cells"])
    assert len(cells) == 4 and len(set(cells)) == 4
    encoded = ";".join(f"{row_value},{col_value}" for row_value, col_value in cells)
    return f"{placement['pieceType']}|{encoded}"


assert len(FEATURE_NAMES) == 16 and set(FEATURE_NAMES).isdisjoint(FORBIDDEN_FEATURES)
training_candidates["candidateIndex"] = np.arange(len(training_candidates), dtype=np.int64)
training_candidates["featureVector"] = [extract_feature_vector(row) for _, row in training_candidates.iterrows()]
training_candidates["equivalenceKey"] = [create_equivalence_key(row) for _, row in training_candidates.iterrows()]
for (_, class_key), members in training_candidates.groupby(["decisionKey", "equivalenceKey"], sort=False):
    features = np.vstack(members["featureVector"])
    assert np.array_equal(features, np.repeat(features[:1], len(features), axis=0)), f"EQUIVALENCE_FEATURE_MISMATCH:{class_key}"
    for field in ["candidateResult", "candidateMetricsAfter", "metricDelta"]:
        encoded = members[field].map(lambda value: json.dumps(value, sort_keys=True, separators=(",", ":")))
        assert encoded.nunique() == 1, f"EQUIVALENCE_RESULT_MISMATCH:{field}"
display(pd.DataFrame({"index": range(16), "feature": FEATURE_NAMES}))


## 6. V4 pair와 weight 진단

**왜 실행하는가:** adopted V4가 selected physical class 대 고유 non-selected class만 한 번씩 비교하는지 확인한다.

**입력:** decision별 equivalence class와 selected candidate.

**계산:** non-selected physical class가 K개면 각 representative pair에 `1/K`를 준다. selected-class alias는 negative pair가 아니다.

**읽는 법:** represented decision의 pair weight 합은 정확히 1이고 physical alternative가 없는 decision은 pairless로 따로 센다.

**보류 조건:** class representative feature가 유일하지 않거나 decision weight 합이 1이 아니면 중단한다.


In [ ]:
def build_v4_pairs(frame: pd.DataFrame, scaled_by_index: dict[int, np.ndarray]) -> tuple[pd.DataFrame, list[str]]:
    records: list[dict[str, Any]] = []
    pairless: list[str] = []
    for decision_key in sorted(frame["decisionKey"].unique(), key=codepoint_key):
        decision = frame.loc[frame["decisionKey"] == decision_key]
        selected = decision.loc[decision["selectedBySource"]].iloc[0]
        physical = decision.loc[(~decision["selectedBySource"]) & (decision["equivalenceKey"] != selected["equivalenceKey"])]
        class_keys = sorted(physical["equivalenceKey"].unique(), key=codepoint_key)
        alternative_class_count = len(class_keys)
        if alternative_class_count == 0:
            pairless.append(decision_key)
            continue
        pair_weight = 1.0 / alternative_class_count
        for class_key in class_keys:
            members = physical.loc[physical["equivalenceKey"] == class_key].sort_values("placementKey", key=lambda column: column.map(codepoint_key))
            representative = members.iloc[0]
            difference = scaled_by_index[int(selected["candidateIndex"])] - scaled_by_index[int(representative["candidateIndex"])]
            records.append({"decisionKey": decision_key, "alternativeClassKey": class_key, "pairWeight": pair_weight, "difference": difference})
    pairs = pd.DataFrame(records)
    assert len(pairs) > 0, "V4_PAIR_REQUIRED"
    weight_by_decision = pairs.groupby("decisionKey", sort=False)["pairWeight"].sum()
    assert np.allclose(weight_by_decision.to_numpy(), 1.0, rtol=0.0, atol=1e-12), "DECISION_BALANCED_PAIR_WEIGHT_REQUIRED"
    return pairs, pairless


## 7. Shared raw-candidate scaler와 symmetric sample

**왜 실행하는가:** V4 class multiplicity가 scaler 통계를 바꾸지 않도록 raw training candidate 전체에 scaler를 fit한다.

**입력:** 선택된 cohort의 16-feature raw candidate.

**계산:** 한 StandardScaler로 candidate를 변환하고 V4 difference의 ± sample에 pair weight 절반씩 배분한다.

**읽는 법:** scaler `n_samples_seen_`는 fitted candidate 수와 같고 symmetric sample weight 합은 represented decision 수와 같다.

**보류 조건:** scaler가 class representative에 fit되거나 transformed weight가 pair 계약과 다르면 중단한다.


In [ ]:
RAW_FEATURES = np.vstack(training_candidates["featureVector"])
SCALER = StandardScaler(with_mean=True, with_std=True)
SCALED_FEATURES = SCALER.fit_transform(RAW_FEATURES)
assert int(SCALER.n_samples_seen_) == len(training_candidates), "RAW_CANDIDATE_SCALER_SCOPE_REQUIRED"
assert np.isfinite(SCALER.mean_).all() and np.isfinite(SCALER.scale_).all()
SCALED_BY_INDEX = {int(index): SCALED_FEATURES[position] for position, index in enumerate(training_candidates["candidateIndex"])}
V4_PAIRS, PAIRLESS_DECISIONS = build_v4_pairs(training_candidates, SCALED_BY_INDEX)


def create_symmetric_samples(pairs: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray, dict[str, Any]]:
    differences = np.vstack(pairs["difference"])
    pair_weights = pairs["pairWeight"].to_numpy(dtype=np.float64)
    x_values = np.empty((len(pairs) * 2, differences.shape[1]), dtype=np.float64)
    labels = np.empty(len(pairs) * 2, dtype=np.int64)
    sample_weights = np.empty(len(pairs) * 2, dtype=np.float64)
    x_values[0::2], x_values[1::2] = differences, -differences
    labels[0::2], labels[1::2] = 1, 0
    sample_weights[0::2], sample_weights[1::2] = pair_weights / 2.0, pair_weights / 2.0
    decision_keys = np.repeat(pairs["decisionKey"].to_numpy(), 2)
    actual = pd.DataFrame({"decisionKey": decision_keys, "weight": sample_weights}).groupby("decisionKey", sort=False)["weight"].sum()
    expected = pairs.groupby("decisionKey", sort=False)["pairWeight"].sum()
    assert np.allclose(actual.loc[expected.index], expected, rtol=0.0, atol=1e-12)
    assert np.allclose(actual.to_numpy(), 1.0, rtol=0.0, atol=1e-12)
    summary = {"symmetricSampleCount": int(len(x_values)), "sampleWeightTotal": float(sample_weights.sum()), "minimumDecisionWeight": float(actual.min()), "maximumDecisionWeight": float(actual.max())}
    return x_values, labels, sample_weights, summary


X_TRAIN, Y_TRAIN, SAMPLE_WEIGHTS, WEIGHT_DIAGNOSTICS = create_symmetric_samples(V4_PAIRS)
display(pd.DataFrame([{"representation": REPRESENTATION, "pairCount": len(V4_PAIRS), "pairlessDecisions": len(PAIRLESS_DECISIONS), **WEIGHT_DIAGNOSTICS}]))


## 8. Development-only batch fit

**왜 실행하는가:** 고정 estimator로 단일 development coefficient를 계산하고 convergence를 진단한다.

**입력:** symmetric V4 samples, actual sample weights와 전체 선택 cohort decision 수.

**계산:** `C=1/(decisionCount×l2)`로 LogisticRegression을 fit하고 weighted training log loss를 optimization 진단으로 기록한다.

**읽는 법:** log loss는 training diagnostic일 뿐 held-out metric이나 gameplay 성능이 아니다.

**보류 조건:** warning, max iteration 도달 또는 non-finite coefficient면 report를 쓰지 않는다.


In [ ]:
C_VALUE = 1.0 / (training_decision_count * MODEL_CONFIG["l2"])
MODEL = LogisticRegression(C=C_VALUE, solver="lbfgs", fit_intercept=False, class_weight=None, tol=1e-12, max_iter=10_000)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    MODEL.fit(X_TRAIN, Y_TRAIN, sample_weight=SAMPLE_WEIGHTS)
convergence_warnings = [str(item.message) for item in caught if issubclass(item.category, ConvergenceWarning)]
unexpected_warnings = [f"{item.category.__name__}:{item.message}" for item in caught if not issubclass(item.category, ConvergenceWarning)]
assert not convergence_warnings, convergence_warnings
assert not unexpected_warnings, unexpected_warnings
assert int(MODEL.n_iter_[0]) < MODEL_CONFIG["maxIterations"]
COEFFICIENTS = MODEL.coef_[0].astype(np.float64)
assert COEFFICIENTS.shape == (16,) and np.isfinite(COEFFICIENTS).all()
probabilities = MODEL.predict_proba(X_TRAIN)[:, 1]
probabilities = np.clip(probabilities, np.finfo(np.float64).eps, 1.0 - np.finfo(np.float64).eps)
losses = -(Y_TRAIN * np.log(probabilities) + (1 - Y_TRAIN) * np.log(1.0 - probabilities))
WEIGHTED_TRAINING_LOG_LOSS = float(np.average(losses, weights=SAMPLE_WEIGHTS))
assert np.isfinite(WEIGHTED_TRAINING_LOG_LOSS)
display(pd.DataFrame([{"runMode": RUN_MODE, "c": C_VALUE, "iterations": int(MODEL.n_iter_[0]), "coefficientL2Norm": float(np.linalg.norm(COEFFICIENTS)), "weightedTrainingLogLoss": WEIGHTED_TRAINING_LOG_LOSS}]))


## 9. 결정적 checkpoint

**왜 실행하는가:** source를 건드리지 않고 scaler, coefficient와 학습 진단을 byte-stable JSON으로 보존한다.

**입력:** 검증된 dataset checksum, cohort, scaler, model과 diagnostics.

**계산:** timestamp와 절대 경로 없이 sort-key compact JSON을 같은 디렉터리의 random exclusive staging file에 쓴 뒤 atomic replace한다.

**읽는 법:** 모든 실행은 `developmentOnly=true`, `productionIntegration=false`, `canonicalResultEligible=false`다.

**보류 조건:** report path나 staging path가 source의 lexical/realpath/inode alias이면 쓰지 않는다. Source checksum이 실행 후 달라도 실패한다.


In [ ]:
def assert_not_source_alias(path: Path, protected_sources: list[Path]) -> None:
    lexical = Path(os.path.abspath(path))
    resolved = path.resolve(strict=False)
    for source in protected_sources:
        if lexical == Path(os.path.abspath(source)) or resolved == source.resolve(strict=True):
            raise AssertionError(f"REPORT_SOURCE_PATH_COLLISION:{source.name}")
        if path.exists() and os.path.samestat(path.stat(), source.stat()):
            raise AssertionError(f"REPORT_SOURCE_INODE_COLLISION:{source.name}")


def atomic_write_text(output_path: Path, text: str, protected_sources: list[Path]) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    assert_not_source_alias(output_path, protected_sources)
    staging_path: Path | None = None
    try:
        for _ in range(100):
            candidate = output_path.parent / f".{output_path.name}.{secrets.token_hex(16)}.tmp"
            assert_not_source_alias(candidate, protected_sources)
            try:
                descriptor = os.open(candidate, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)  # exclusive create
                staging_path = candidate
                break
            except FileExistsError:
                continue
        assert staging_path is not None, "EXCLUSIVE_STAGING_CREATE_FAILED"
        with os.fdopen(descriptor, "w", encoding="utf-8", newline="") as handle:
            handle.write(text)
            handle.flush()
            os.fsync(handle.fileno())
        assert_not_source_alias(staging_path, protected_sources)
        os.replace(staging_path, output_path)
        staging_path = None
    finally:
        if staging_path is not None and staging_path.exists():
            staging_path.unlink()


source_checksums_after = {name: sha256_file(DATASET / name) for name in DATASET_FILES}
assert source_checksums_after == source_checksums_before, "SOURCE_DATASET_MUTATED"
CHECKPOINT = {
    "experimentId": EXPERIMENT_ID,
    "status": "development_checkpoint_created" if FULL_COHORT_COVERED else "awaiting_user_full_run",
    "runMode": RUN_MODE,
    "fullCohortCovered": FULL_COHORT_COVERED,
    "developmentOnly": True,
    "productionIntegration": False,
    "canonicalResultEligible": False,
    "policyDecisionEligible": False,
    "heldOutEvidence": False,
    "evaluationMetricsIncluded": False,
    "dataset": {"relativePath": DATASET_RELATIVE.as_posix(), "checksumsBefore": source_checksums_before, "checksumsAfter": source_checksums_after, "manifestLifecycleGameCount": int(MANIFEST["totalGameCount"]), "availableAcceptedGameCount": len(ALL_GAME_IDS), "availableDecisionCount": int(len(decision_contract)), "availableCandidateCount": int(len(all_candidates))},
    "trainingCohort": {"gameCount": len(TRAINING_GAME_IDS), "decisionCount": training_decision_count, "candidateCount": int(len(training_candidates)), "representedDecisionCount": int(V4_PAIRS["decisionKey"].nunique()), "gameIdSha256": hashlib.sha256("\n".join(TRAINING_GAME_IDS).encode("utf-8")).hexdigest()},
    "featureContract": {"featureNames": FEATURE_NAMES, "forbiddenLeakageInputs": sorted(FORBIDDEN_FEATURES), "scalerFitScope": "selected training cohort raw candidates only", "scalerFitCandidateCount": int(SCALER.n_samples_seen_), "mean": [float(value) for value in SCALER.mean_], "scale": [float(value) for value in SCALER.scale_]},
    "representation": {"name": REPRESENTATION, "unit": "occupied-cell physical class", "pairWeight": "one over unique non-selected physical classes", "selectedClassAliasesExcluded": True, "pairCount": int(len(V4_PAIRS)), "pairlessDecisionCount": len(PAIRLESS_DECISIONS)},
    "estimator": {**MODEL_CONFIG, "c": C_VALUE},
    "modelCheckpoint": {"coefficientFeatureOrder": FEATURE_NAMES, "coefficients": [float(value) for value in COEFFICIENTS], "coefficientL2Norm": float(np.linalg.norm(COEFFICIENTS)), "coefficientMaxAbsolute": float(np.max(np.abs(COEFFICIENTS))), "intercept": False},
    "trainingDiagnostics": {**WEIGHT_DIAGNOSTICS, "iterations": int(MODEL.n_iter_[0]), "convergenceWarnings": convergence_warnings, "unexpectedWarnings": unexpected_warnings, "weightedTrainingLogLoss": WEIGHTED_TRAINING_LOG_LOSS, "weightedTrainingLogLossRole": "optimization diagnostic only; not held-out performance"},
    "environment": ENVIRONMENT,
}
REPORT_TEXT = json.dumps(CHECKPOINT, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False) + "\n"
atomic_write_text(REPORT_PATH, REPORT_TEXT, [DATASET / name for name in DATASET_FILES])
assert REPORT_PATH.read_text(encoding="utf-8") == REPORT_TEXT
display(pd.DataFrame([{"status": CHECKPOINT["status"], "runMode": RUN_MODE, "fullCohortCovered": FULL_COHORT_COVERED, "reportSha256": hashlib.sha256(REPORT_TEXT.encode("utf-8")).hexdigest()}]))


## 10. 사용자 full-run gate

Smoke가 통과하면 구현 경로만 준비된 것이다. 다음 checkpoint는 사용자가 `PLACEMENT_V4_MAX_TRAINING_GAMES=75`로 full 실행하는 것이다. Full report도 `developmentOnly=true`, `productionIntegration=false`, `canonicalResultEligible=false`, `heldOutEvidence=false`를 유지한다.

Full 결과로 말할 수 있는 것은 cumulative cohort에 대한 deterministic batch fit과 optimization 진단뿐이다. 일반화, held-out 성능, 승률, gameplay promotion 또는 production readiness는 말할 수 없다.
